# Task 3 — Model Validation, Overfitting Control & Hyperparameter Tuning
**Dataset:** California Housing (`housing(1).csv`)  
**Target:** `median_house_value`

This notebook compares Linear Regression, Ridge Regression, a baseline Decision Tree, and a tuned Decision Tree. It uses an 80/20 train-test split, 5-fold cross-validation, and GridSearchCV. Put `housing(1).csv` in the same folder as this notebook before running.

In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42

In [ ]:
# Step 2: Load and inspect the provided dataset
df = pd.read_csv("housing(1).csv")
print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values per column:\\n", df.isnull().sum())
print("\\nColumns:", df.columns.tolist())

In [ ]:
# Step 3: Separate features and target
target = "median_house_value"
X = df.drop(columns=[target])
y = df[target]

# Handle numeric and categorical columns separately. This also handles the
# missing values in total_bedrooms and encodes ocean_proximity.
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## Step 4: Detect overfitting with train vs. test performance
A large gap between training and test error indicates possible overfitting. The unpruned Decision Tree can fit the training data extremely closely, so we compare it with simpler models and later tune its complexity.

In [ ]:
# Step 5: Baseline model evaluation with 5-fold cross-validation
model_definitions = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree (baseline)": DecisionTreeRegressor(random_state=RANDOM_STATE)
}

comparison_rows = []
fitted_models = {}
for name, estimator in model_definitions.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    cv_rmse = -cross_val_score(
        pipe, X_train, y_train,
        scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
    )
    comparison_rows.append({
        "Model": name,
        "Train RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
        "Test RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "Test R2": r2_score(y_test, test_pred),
        "CV RMSE Mean": cv_rmse.mean(),
        "CV RMSE Std": cv_rmse.std()
    })

baseline_results = pd.DataFrame(comparison_rows)
display(baseline_results.round(2))

## Step 6: Hyperparameter tuning with GridSearchCV
`max_depth` limits tree depth; `min_samples_split` specifies the minimum samples required to split a node; `min_samples_leaf` sets the minimum samples allowed in a leaf. Five-fold CV selects the combination with the lowest average RMSE.

In [ ]:
param_grid = {
    "model__max_depth": [3, 5, 7, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5]
}
tree_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))
])
grid = GridSearchCV(
    tree_pipeline, param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5, n_jobs=-1, return_train_score=True
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best 5-fold CV RMSE:", round(-grid.best_score_, 2))
best_tree = grid.best_estimator_

In [ ]:
# Step 7: Evaluate the optimized model on the held-out test set
train_pred = best_tree.predict(X_train)
test_pred = best_tree.predict(X_test)

tuned_row = {
    "Model": "Tuned Decision Tree",
    "Train RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
    "Test RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
    "Test R2": r2_score(y_test, test_pred),
    "CV RMSE Mean": -grid.best_score_,
    "CV RMSE Std": grid.cv_results_["std_test_score"][grid.best_index_]
}
all_results = pd.concat([baseline_results, pd.DataFrame([tuned_row])], ignore_index=True)
display(all_results.round(2))

all_results.to_csv("Task3_Model_Comparison.csv", index=False)
tuning_results = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
tuning_results[[
    "param_model__max_depth", "param_model__min_samples_split",
    "param_model__min_samples_leaf", "mean_test_score",
    "std_test_score", "rank_test_score"
]].to_csv("Task3_Hyperparameter_Tuning_Results.csv", index=False)
print("Saved model comparison and tuning results CSV files.")

In [ ]:
# Step 8: Visualize test RMSE and R² for model comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
all_results.plot(x="Model", y="Test RMSE", kind="bar", ax=axes[0], legend=False, title="Test RMSE (lower is better)")
all_results.plot(x="Model", y="Test R2", kind="bar", ax=axes[1], legend=False, title="Test R² (higher is better)")
axes[0].set_ylabel("RMSE in house-value units")
axes[1].set_ylabel("R²")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=25)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Step 9: Final model selection justification
Select the model using held-out test performance together with cross-validation, not training score alone. In this run, the tuned Decision Tree has the lowest test RMSE and highest test R² among the evaluated models. Its training RMSE is no longer zero, indicating that restricting tree complexity reduced the extreme overfitting of the baseline tree. Cross-validation also gives a lower average RMSE for the tuned tree than the baseline tree.

**Caution:** the target is capped in the California Housing dataset, and scores depend on the exact data file and preprocessing. Do not use the test set repeatedly for tuning; keep it as a final, held-out evaluation.